In [1]:
import pandas as pd
import numpy as np
import psycopg
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"

In [ ]:
#TODO: set up a script that will get the new weekly hero stats from stratz

In [ ]:
## Calculate form: for now, wins out of last 5 games, simple int
query = 'SELECT * FROM match_details;'
df = pd.DataFrame(dbf.query_select_to_df(conn_str, query, table_name='match_details'))

In [3]:
df['radiantForm'] = pd.NA
df['direForm'] = pd.NA

In [20]:
df = df.sort_values(by='startDateTime', ascending=False)
for idx, row in df.iterrows():
    tmp_rad_df = df[((df['radiantTeamId'] == row['radiantTeamId']) | (df['direTeamId'] == row['radiantTeamId'])) & (df['startDateTime'] < row['startDateTime'])]
    rad_wins = 0
    tmp_dire_df = df[((df['radiantTeamId'] == row['direTeamId']) | (df['direTeamId'] == row['direTeamId'])) & (df['startDateTime'] < row['startDateTime'])]
    dire_wins = 0
    for tmp_idx, tmp_row in tmp_rad_df.head(5).iterrows():
        if tmp_row['radiantTeamId'] == row['radiantTeamId']:
            result = 1 if tmp_row['didRadiantWin'] else 0
        elif tmp_row['direTeamId'] == row['radiantTeamId']:
            result = 0 if tmp_row['didRadiantWin'] else 1
        rad_wins += result
    for tmp_idx, tmp_row in tmp_dire_df.head(5).iterrows():
        if tmp_row['radiantTeamId'] == row['direTeamId']:
            result = 1 if tmp_row['didRadiantWin'] else 0
        elif tmp_row['direTeamId'] == row['direTeamId']:
            result = 0 if tmp_row['didRadiantWin'] else 1
        dire_wins += result
    df.loc[idx, 'radiantForm'] = rad_wins
    df.loc[idx, 'direForm'] = dire_wins

In [21]:
df

,id,tournamentId,tournamentRound,leagueId,radiantTeamId,direTeamId,seriesId,gameVersionId,regionId,clusterId,...,averageRank,averageImp,bracket,analysisOutcome,topLaneOutcome,midLaneOutcome,bottomLaneOutcome,predictedOutcomeWeight,radiantForm,direForm
2765,8636567078,None,None,19148,8261500,726228,1052680,182,17,225,...,None,-11,8,COMEBACK,TIE,DIRE_VICTORY,RADIANT_VICTORY,38,4,4
2291,8636456043,None,None,19148,8261500,726228,1052680,182,17,225,...,None,-12,8,NONE,DIRE_VICTORY,TIE,RADIANT_VICTORY,45,5,3
2290,8636156992,None,None,19148,726228,10008067,1052643,182,17,225,...,None,-10,8,NONE,DIRE_VICTORY,TIE,TIE,53,4,4
2764,8636084693,None,None,19148,726228,10008067,1052643,182,17,225,...,None,-12,8,NONE,RADIANT_VICTORY,TIE,RADIANT_VICTORY,47,3,5
2289,8636033433,None,None,19148,9579337,8261500,1052627,182,17,225,...,None,-8,8,NONE,TIE,TIE,TIE,59,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2,8184093905,None,None,17765,9572001,9247354,951544,179,3,274,...,None,0,8,NONE,TIE,DIRE_VICTORY,TIE,31,1,0
542,8183945792,None,None,17765,9572001,9247354,951544,179,3,273,...,None,-2,8,NONE,DIRE_VICTORY,TIE,RADIANT_VICTORY,46,0,0
1,8183837907,None,None,17765,2163,8255888,951498,179,3,273,...,None,-1,8,NONE,DIRE_VICTORY,DIRE_VICTORY,RADIANT_VICTORY,58,1,1
28,8183722972,None,None,17765,8255888,2163,951498,179,3,273,...,None,-10,8,NONE,TIE,TIE,TIE,53,0,1


In [18]:
tmp_row

'd'

In [15]:
tmp_df.columns

Index(['id', 'tournamentId', 'tournamentRound', 'leagueId', 'radiantTeamId',
       'direTeamId', 'seriesId', 'gameVersionId', 'regionId', 'clusterId',
       'didRadiantWin', 'startDateTime', 'endDateTime', 'durationSeconds',
       'firstBloodTime', 'towerStatusRadiant', 'towerStatusDire',
       'barracksStatusRadiant', 'barracksStatusDire', 'rank', 'actualRank',
       'averageRank', 'averageImp', 'bracket', 'analysisOutcome',
       'topLaneOutcome', 'midLaneOutcome', 'bottomLaneOutcome',
       'predictedOutcomeWeight', 'radiantForm', 'direForm'],
      dtype='object')